In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# Step10 Immune Infiltration Analysis
# Step10.1 Initialize
# ============================================================

import os
import sys
import numpy as np
import pandas as pd

print("Python:")
print(sys.executable)

project_dir = _hcc_path('')

step11_dir = os.path.join(
    project_dir,
    "LIHC_RNA_Step11_Immune"
)

input_dir = os.path.join(
    step11_dir,
    "Input"
)

result_dir = os.path.join(
    step11_dir,
    "Results"
)

fig_dir = os.path.join(
    step11_dir,
    "Figures"
)

for d in [
    step11_dir,
    input_dir,
    result_dir,
    fig_dir
]:
    os.makedirs(
        d,
        exist_ok=True
    )

print("\nStep11 directory:")
print(step11_dir)

In [ ]:
# ============================================================
# Step10.2 Load normalized expression matrix
# ============================================================

rna_file = os.path.join(
    project_dir,
    "LIHC_RNA_Step1",
    "LIHC_RNA_Normalized_371.tsv"
)

rna = pd.read_csv(
    rna_file,
    sep="\t",
    index_col=0
)

print("RNA shape:", rna.shape)

display(
    rna.iloc[:5, :5]
)

In [ ]:
# ============================================================
# Step10.3 Load final RiskScore
# ============================================================

risk_file = os.path.join(
    project_dir,
    "LIHC_RNA_Step8_Final_Model",
    "Results",
    "Final_Top10_RiskScore.csv"
)

risk = pd.read_csv(
    risk_file
)

print("Risk shape:", risk.shape)
print(risk.columns.tolist())

display(
    risk.head()
)

In [ ]:
# ============================================================
# Step10.4 Define High / Low risk groups
# ============================================================

risk_cutoff = risk[
    "RiskScore"
].median()

risk["RiskGroup"] = np.where(
    risk["RiskScore"] > risk_cutoff,
    "High",
    "Low"
)

print(
    risk["RiskGroup"].value_counts()
)

print(
    "\nMedian RiskScore cutoff:",
    risk_cutoff
)

In [ ]:
# ============================================================
# Step10.5 Match expression and RiskScore
# ============================================================

common_samples = [
    sample
    for sample in risk["Sample_ID"]
    if sample in rna.columns
]

rna_immune = rna[
    common_samples
].copy()

risk_immune = (
    risk
    .set_index("Sample_ID")
    .loc[common_samples]
    .copy()
)

print(
    "RNA immune:",
    rna_immune.shape
)

print(
    "Risk immune:",
    risk_immune.shape
)

print(
    "RNA samples:",
    len(rna_immune.columns)
)

print(
    "Risk samples:",
    len(risk_immune.index)
)

print(
    "Sample order identical:",
    list(rna_immune.columns)
    ==
    list(risk_immune.index)
)

In [ ]:
# ============================================================
# Step10.6 QC
# ============================================================

print(
    "Duplicated genes:",
    rna_immune.index.duplicated().sum()
)

print(
    "Missing expression values:",
    rna_immune.isna().sum().sum()
)

print(
    "Min expression:",
    rna_immune.min().min()
)

print(
    "Max expression:",
    rna_immune.max().max()
)

print(
    "Duplicated samples:",
    pd.Index(
        rna_immune.columns
    ).duplicated().sum()
)

print(
    "\nRiskGroup:"
)

print(
    risk_immune[
        "RiskGroup"
    ].value_counts()
)

In [ ]:
# ============================================================
# Step10.7 Save standardized Step10 input
# ============================================================

expression_output = os.path.join(
    input_dir,
    "LIHC_Expression_365_log2CPM.tsv"
)

risk_output = os.path.join(
    input_dir,
    "LIHC_RiskGroup_365.csv"
)

rna_immune.to_csv(
    expression_output,
    sep="\t"
)

risk_immune.to_csv(
    risk_output
)

print("Saved expression:")
print(expression_output)

print("\nSaved risk information:")
print(risk_output)

In [ ]:
# ============================================================
# Step10.8 Prepare ESTIMATE input
# ============================================================

estimate_input = os.path.join(
    input_dir,
    "LIHC_ESTIMATE_Input_365.tsv"
)

rna_immune.to_csv(
    estimate_input,
    sep="\t"
)

print("ESTIMATE input saved:")
print(estimate_input)

In [ ]:
#中间跳转11.2特殊环境

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# Step11.12 Load ESTIMATE results in TCGA environment
# ============================================================

import os
import pandas as pd
import numpy as np

project_dir = _hcc_path('')

step11_dir = os.path.join(
    project_dir,
    "LIHC_RNA_Step11_Immune"
)

input_dir = os.path.join(
    step11_dir,
    "Input"
)

result_dir = os.path.join(
    step11_dir,
    "Results"
)

fig_dir = os.path.join(
    step11_dir,
    "Figures"
)


# ESTIMATE scores
estimate_score = pd.read_csv(
    os.path.join(
        result_dir,
        "LIHC_ESTIMATE_Scores_365.csv"
    ),
    index_col=0
)


# Risk group
risk_immune = pd.read_csv(
    os.path.join(
        input_dir,
        "LIHC_RiskGroup_365.csv"
    ),
    index_col=0
)


print("ESTIMATE:", estimate_score.shape)
print("Risk:", risk_immune.shape)

print("\nESTIMATE columns:")
print(estimate_score.columns.tolist())

print("\nRisk columns:")
print(risk_immune.columns.tolist())

In [ ]:
# ============================================================
# Step11.13 Merge ESTIMATE with RiskScore
# ============================================================

estimate_score.index.name = "Sample_ID"
risk_immune.index.name = "Sample_ID"


estimate_merge = (
    estimate_score
    .reset_index()
    .merge(
        risk_immune.reset_index(),
        on="Sample_ID",
        how="inner"
    )
)


print("Merged shape:", estimate_merge.shape)

print(
    "Unique samples:",
    estimate_merge["Sample_ID"].nunique()
)

display(
    estimate_merge.head()
)

In [ ]:
# ============================================================
# Step11.14 ESTIMATE merged-data QC
# ============================================================

check_cols = [
    "RiskScore",
    "RiskGroup",
    "EST_stromal",
    "EST_immune",
    "estimate"
]

print(
    estimate_merge[
        check_cols
    ].isna().sum()
)

print("\nRisk groups:")

print(
    estimate_merge[
        "RiskGroup"
    ].value_counts()
)

In [ ]:
# ============================================================
# Step11.15 High vs Low risk comparison
# ============================================================

from scipy.stats import mannwhitneyu


score_cols = [
    "EST_stromal",
    "EST_immune",
    "estimate"
]


comparison_results = []


for col in score_cols:

    high = estimate_merge.loc[
        estimate_merge["RiskGroup"] == "High",
        col
    ].dropna()

    low = estimate_merge.loc[
        estimate_merge["RiskGroup"] == "Low",
        col
    ].dropna()


    stat, p = mannwhitneyu(
        high,
        low,
        alternative="two-sided"
    )


    comparison_results.append({
        "Score": col,

        "High_n": len(high),
        "Low_n": len(low),

        "High_mean": high.mean(),
        "Low_mean": low.mean(),

        "High_median": high.median(),
        "Low_median": low.median(),

        "U_statistic": stat,
        "P_value": p
    })


comparison_results = pd.DataFrame(
    comparison_results
)


display(
    comparison_results
)

In [ ]:
# ============================================================
# Step11.16 Correlation with continuous RiskScore
# ============================================================

from scipy.stats import spearmanr


correlation_results = []


for col in score_cols:

    tmp = estimate_merge[
        [
            "RiskScore",
            col
        ]
    ].dropna()


    rho, p = spearmanr(
        tmp["RiskScore"],
        tmp[col]
    )


    correlation_results.append({
        "Score": col,
        "N": len(tmp),
        "Spearman_rho": rho,
        "P_value": p
    })


correlation_results = pd.DataFrame(
    correlation_results
)


display(
    correlation_results
)

In [ ]:
# ============================================================
# Step11.17 Save ESTIMATE statistical results
# ============================================================

estimate_merge.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ESTIMATE_With_RiskGroup_365.csv"
    ),
    index=False
)


comparison_results.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ESTIMATE_High_vs_Low.csv"
    ),
    index=False
)


correlation_results.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ESTIMATE_RiskScore_Correlation.csv"
    ),
    index=False
)


print("ESTIMATE statistical results saved.")

In [ ]:
# ============================================================
# Step11.18 Multiple-testing correction
# ============================================================

from statsmodels.stats.multitest import multipletests


comparison_results["FDR"] = multipletests(
    comparison_results["P_value"],
    method="fdr_bh"
)[1]


correlation_results["FDR"] = multipletests(
    correlation_results["P_value"],
    method="fdr_bh"
)[1]


display(comparison_results)
display(correlation_results)

In [ ]:
comparison_results.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ESTIMATE_High_vs_Low.csv"
    ),
    index=False
)

correlation_results.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ESTIMATE_RiskScore_Correlation.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# Step11.19 ESTIMATE High vs Low boxplots
# ============================================================

import matplotlib.pyplot as plt
import numpy as np


plot_info = {
    "EST_stromal": "Stromal score",
    "EST_immune": "Immune score",
    "estimate": "ESTIMATE score"
}


def p_label(p):
    if p < 0.001:
        return "P < 0.001"
    return f"P = {p:.3f}"


for col, ylabel in plot_info.items():

    low = estimate_merge.loc[
        estimate_merge["RiskGroup"] == "Low",
        col
    ].to_numpy(dtype=float)

    high = estimate_merge.loc[
        estimate_merge["RiskGroup"] == "High",
        col
    ].to_numpy(dtype=float)

    p = comparison_results.loc[
        comparison_results["Score"] == col,
        "P_value"
    ].iloc[0]


    fig, ax = plt.subplots(
        figsize=(5.5, 5.5)
    )


    ax.boxplot(
        [low, high],
        tick_labels=[
            "Low risk",
            "High risk"
        ],
        widths=0.55,
        showfliers=False
    )


    rng = np.random.default_rng(123)

    for i, values in enumerate(
        [low, high],
        start=1
    ):

        x = rng.normal(
            i,
            0.04,
            len(values)
        )

        ax.scatter(
            x,
            values,
            s=9,
            alpha=0.25
        )


    ymin = min(low.min(), high.min())
    ymax = max(low.max(), high.max())
    yrange = ymax - ymin

    y = ymax + 0.05 * yrange


    ax.plot(
        [1, 1, 2, 2],
        [
            y,
            y + 0.02 * yrange,
            y + 0.02 * yrange,
            y
        ],
        linewidth=1
    )


    ax.text(
        1.5,
        y + 0.03 * yrange,
        p_label(p),
        ha="center",
        fontsize=11
    )


    ax.set_ylabel(
        ylabel,
        fontsize=12
    )

    ax.set_title(
        ylabel,
        fontsize=13
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


    fig.tight_layout()


    fig.savefig(
        os.path.join(
            fig_dir,
            f"{col}_High_vs_Low.png"
        ),
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(
            fig_dir,
            f"{col}_High_vs_Low.pdf"
        ),
        bbox_inches="tight"
    )


    plt.show()
    plt.close(fig)

In [ ]:
# ============================================================
# Step11.20 Combined ESTIMATE boxplots
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(14, 4.8)
)


rng = np.random.default_rng(123)


for ax, (col, ylabel) in zip(
    axes,
    plot_info.items()
):

    low = estimate_merge.loc[
        estimate_merge["RiskGroup"] == "Low",
        col
    ].to_numpy(dtype=float)

    high = estimate_merge.loc[
        estimate_merge["RiskGroup"] == "High",
        col
    ].to_numpy(dtype=float)

    p = comparison_results.loc[
        comparison_results["Score"] == col,
        "P_value"
    ].iloc[0]


    ax.boxplot(
        [low, high],
        tick_labels=[
            "Low risk",
            "High risk"
        ],
        widths=0.55,
        showfliers=False
    )


    for i, values in enumerate(
        [low, high],
        start=1
    ):

        x = rng.normal(
            i,
            0.035,
            len(values)
        )

        ax.scatter(
            x,
            values,
            s=7,
            alpha=0.20
        )


    ymin = min(low.min(), high.min())
    ymax = max(low.max(), high.max())
    yrange = ymax - ymin

    y = ymax + 0.05 * yrange


    ax.plot(
        [1, 1, 2, 2],
        [
            y,
            y + 0.02 * yrange,
            y + 0.02 * yrange,
            y
        ],
        linewidth=1
    )


    ax.text(
        1.5,
        y + 0.03 * yrange,
        p_label(p),
        ha="center",
        fontsize=10
    )


    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ESTIMATE_High_vs_Low_Combined.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ESTIMATE_High_vs_Low_Combined.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.21 RiskScore correlation plots
# ============================================================

for col, ylabel in plot_info.items():

    x = estimate_merge[
        "RiskScore"
    ].to_numpy(dtype=float)

    y = estimate_merge[
        col
    ].to_numpy(dtype=float)


    rho = correlation_results.loc[
        correlation_results["Score"] == col,
        "Spearman_rho"
    ].iloc[0]

    p = correlation_results.loc[
        correlation_results["Score"] == col,
        "P_value"
    ].iloc[0]


    coef = np.polyfit(
        x,
        y,
        1
    )

    xline = np.linspace(
        x.min(),
        x.max(),
        200
    )

    yline = (
        coef[0] * xline
        +
        coef[1]
    )


    fig, ax = plt.subplots(
        figsize=(5.5, 5.5)
    )


    ax.scatter(
        x,
        y,
        s=20,
        alpha=0.45
    )

    ax.plot(
        xline,
        yline,
        linewidth=2
    )


    ax.text(
        0.05,
        0.95,
        f"Spearman ρ = {rho:.3f}\n{p_label(p)}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=11
    )


    ax.set_xlabel(
        "Risk score",
        fontsize=12
    )

    ax.set_ylabel(
        ylabel,
        fontsize=12
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


    fig.tight_layout()


    fig.savefig(
        os.path.join(
            fig_dir,
            f"RiskScore_vs_{col}.png"
        ),
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        os.path.join(
            fig_dir,
            f"RiskScore_vs_{col}.pdf"
        ),
        bbox_inches="tight"
    )


    plt.show()
    plt.close(fig)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# Step11.23 Load standardized immune expression matrix
# ============================================================

import os
import pandas as pd
import numpy as np

project_dir = _hcc_path('')

step11_dir = os.path.join(
    project_dir,
    "LIHC_RNA_Step11_Immune"
)

input_dir = os.path.join(
    step11_dir,
    "Input"
)

result_dir = os.path.join(
    step11_dir,
    "Results"
)

fig_dir = os.path.join(
    step11_dir,
    "Figures"
)


rna_immune = pd.read_csv(
    os.path.join(
        input_dir,
        "LIHC_Expression_365_log2CPM.tsv"
    ),
    sep="\t",
    index_col=0
)

risk_immune = pd.read_csv(
    os.path.join(
        input_dir,
        "LIHC_RiskGroup_365.csv"
    ),
    index_col=0
)

print("RNA:", rna_immune.shape)
print("Risk:", risk_immune.shape)

print(
    "Sample order identical:",
    list(rna_immune.columns)
    ==
    list(risk_immune.index)
)

In [ ]:
# ============================================================
# Step11.24 Immune-cell signatures
# ============================================================

immune_signatures = {

    "B_cells": [
        "CD19", "MS4A1", "CD79A", "CD79B", "CD37", "CD22"
    ],

    "CD8_T_cells": [
        "CD8A", "CD8B", "CCL5", "GZMK"
    ],

    "CD4_T_cells": [
        "CD3D", "CD3E", "CD4", "IL7R", "LTB"
    ],

    "Treg": [
        "FOXP3", "IL2RA", "CTLA4", "IKZF2", "TIGIT"
    ],

    "NK_cells": [
        "NKG7", "GNLY", "KLRD1", "KLRK1",
        "PRF1", "XCL1", "XCL2"
    ],

    "Cytotoxic_cells": [
        "NKG7", "GNLY", "GZMA", "GZMB",
        "GZMH", "PRF1", "CTSW"
    ],

    "Macrophages": [
        "CD68", "CD163", "CSF1R", "MSR1",
        "MARCO", "FCGR1A"
    ],

    "M1_macrophages": [
        "NOS2", "IL1B", "TNF", "CXCL9",
        "CXCL10", "CD80"
    ],

    "M2_macrophages": [
        "CD163", "MRC1", "MSR1", "CCL18",
        "IL10", "MARCO"
    ],

    "Dendritic_cells": [
        "CD1C", "CLEC10A", "FCER1A",
        "ITGAX", "CD209"
    ],

    "Neutrophils": [
        "FCGR3B", "CSF3R", "FPR1",
        "CXCR2", "S100A8", "S100A9"
    ],

    "Mast_cells": [
        "TPSAB1", "TPSB2", "KIT",
        "CMA1", "MS4A2"
    ]
}


print("Number of immune signatures:", len(immune_signatures))

for name, genes in immune_signatures.items():
    print(name, len(genes))

In [ ]:
# ============================================================
# Step11.25 Signature coverage QC
# ============================================================

coverage_results = []


for cell_type, genes in immune_signatures.items():

    present = [
        g for g in genes
        if g in rna_immune.index
    ]

    missing = [
        g for g in genes
        if g not in rna_immune.index
    ]

    coverage_results.append({
        "Cell_type": cell_type,
        "Signature_genes": len(genes),
        "Present_genes": len(present),
        "Missing_genes": len(missing),
        "Coverage": len(present) / len(genes)
    })


coverage_df = pd.DataFrame(
    coverage_results
)

display(coverage_df)

In [ ]:
coverage_df.to_csv(
    os.path.join(
        result_dir,
        "ssGSEA_Immune_Signature_Coverage.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# Step11.26 Run ssGSEA
# ============================================================

ssgsea_res = gp.ssgsea(
    data=rna_immune,
    gene_sets=immune_signatures,
    sample_norm_method="rank",
    outdir=None,
    no_plot=True,
    min_size=2,
    max_size=500,
    permutation_num=0,
    threads=4,
    seed=123,
    verbose=True
)

In [ ]:
print(type(ssgsea_res))

print(
    ssgsea_res.res2d.shape
)

display(
    ssgsea_res.res2d.head()
)

In [ ]:
# ============================================================
# Step11.27-11.28
# Build ssGSEA matrix + save
# ============================================================

import os
import pandas as pd


# 1. 检查 ssGSEA 原始结果是否存在
print("ssgsea_res exists:", "ssgsea_res" in globals())

if "ssgsea_res" not in globals():
    raise RuntimeError(
        "ssgsea_res 不存在，请先重新运行 Step11.26 ssGSEA"
    )


# 2. 获取长格式结果
ssgsea_long = ssgsea_res.res2d.copy()

print("Long result shape:", ssgsea_long.shape)
print("Columns:", ssgsea_long.columns.tolist())


# 3. 确保 NES 是数值
ssgsea_long["NES"] = pd.to_numeric(
    ssgsea_long["NES"],
    errors="coerce"
)

print(
    "Missing NES:",
    ssgsea_long["NES"].isna().sum()
)


# 4. 转换成：
# Sample × Immune cell
ssgsea_matrix = (
    ssgsea_long
    .pivot(
        index="Name",
        columns="Term",
        values="NES"
    )
)

ssgsea_matrix.index.name = "Sample_ID"


# 5. 检查结果
print("\nssGSEA matrix shape:")
print(ssgsea_matrix.shape)

print("\nImmune cell types:")
print(ssgsea_matrix.columns.tolist())

display(
    ssgsea_matrix.head()
)


# 6. 保存
matrix_file = os.path.join(
    result_dir,
    "LIHC_ssGSEA_Immune_Cell_Scores_365.csv"
)

long_file = os.path.join(
    result_dir,
    "LIHC_ssGSEA_Immune_Cell_Long.csv"
)


ssgsea_matrix.to_csv(
    matrix_file
)

ssgsea_long.to_csv(
    long_file,
    index=False
)


print("\nSaved successfully:")
print(matrix_file)
print(long_file)

In [ ]:
# ============================================================
# Step11.29 Merge ssGSEA scores with RiskGroup
# ============================================================

# 如果当前内存里没有，就重新读取
if "ssgsea_matrix" not in globals():
    ssgsea_matrix = pd.read_csv(
        os.path.join(
            result_dir,
            "LIHC_ssGSEA_Immune_Cell_Scores_365.csv"
        ),
        index_col=0
    )

if "risk_immune" not in globals():
    risk_immune = pd.read_csv(
        os.path.join(
            input_dir,
            "LIHC_RiskGroup_365.csv"
        ),
        index_col=0
    )

ssgsea_matrix.index.name = "Sample_ID"
risk_immune.index.name = "Sample_ID"


immune_merge = (
    ssgsea_matrix
    .reset_index()
    .merge(
        risk_immune.reset_index(),
        on="Sample_ID",
        how="inner"
    )
)

print("Merged shape:", immune_merge.shape)
print("Unique samples:", immune_merge["Sample_ID"].nunique())

display(immune_merge.head())

In [ ]:
# ============================================================
# Step11.30 High vs Low immune-cell comparison
# ============================================================

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

immune_cols = ssgsea_matrix.columns.tolist()

immune_comparison = []

for cell in immune_cols:

    high = immune_merge.loc[
        immune_merge["RiskGroup"] == "High",
        cell
    ].dropna()

    low = immune_merge.loc[
        immune_merge["RiskGroup"] == "Low",
        cell
    ].dropna()

    stat, p = mannwhitneyu(
        high,
        low,
        alternative="two-sided"
    )

    immune_comparison.append({
        "Cell_type": cell,
        "High_n": len(high),
        "Low_n": len(low),
        "High_mean": high.mean(),
        "Low_mean": low.mean(),
        "High_median": high.median(),
        "Low_median": low.median(),
        "High_minus_Low": high.median() - low.median(),
        "U_statistic": stat,
        "P_value": p
    })


immune_comparison = pd.DataFrame(
    immune_comparison
)


# FDR correction
immune_comparison["FDR"] = multipletests(
    immune_comparison["P_value"],
    method="fdr_bh"
)[1]


# 按FDR排序
immune_comparison = immune_comparison.sort_values(
    ["FDR", "P_value"]
).reset_index(drop=True)


display(immune_comparison)

In [ ]:
# ============================================================
# Step11.31 Save immune-cell comparison
# ============================================================

immune_comparison.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ssGSEA_High_vs_Low_ImmuneCells.csv"
    ),
    index=False
)

immune_merge.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ssGSEA_With_RiskGroup_365.csv"
    ),
    index=False
)

print("Saved.")

In [ ]:
# ============================================================
# Step11.32 RiskScore correlation with immune cells
# ============================================================

from scipy.stats import spearmanr

immune_correlations = []

for cell in immune_cols:

    tmp = immune_merge[
        ["RiskScore", cell]
    ].dropna()

    rho, p = spearmanr(
        tmp["RiskScore"],
        tmp[cell]
    )

    immune_correlations.append({
        "Cell_type": cell,
        "N": len(tmp),
        "Spearman_rho": rho,
        "P_value": p
    })


immune_correlations = pd.DataFrame(
    immune_correlations
)


immune_correlations["FDR"] = multipletests(
    immune_correlations["P_value"],
    method="fdr_bh"
)[1]


immune_correlations = immune_correlations.sort_values(
    ["FDR", "P_value"]
).reset_index(drop=True)


display(immune_correlations)

In [ ]:
immune_correlations.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ssGSEA_RiskScore_ImmuneCell_Correlation.csv"
    ),
    index=False
)

print("Saved.")

In [ ]:
# ============================================================
# Step11.33
# High vs Low risk immune-cell comparison
# Combined boxplots
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# 按你目前FDR从小到大排列
plot_cells = (
    immune_comparison
    .sort_values("FDR")
    ["Cell_type"]
    .tolist()
)


# 整理成长格式
plot_data = immune_merge[
    ["Sample_ID", "RiskGroup"] + plot_cells
].melt(
    id_vars=["Sample_ID", "RiskGroup"],
    value_vars=plot_cells,
    var_name="Cell_type",
    value_name="ssGSEA_score"
)


fig, ax = plt.subplots(
    figsize=(14, 6.5)
)


positions = np.arange(
    len(plot_cells)
)


width = 0.30

rng = np.random.default_rng(123)


for i, cell in enumerate(plot_cells):

    low = plot_data.loc[
        (plot_data["Cell_type"] == cell) &
        (plot_data["RiskGroup"] == "Low"),
        "ssGSEA_score"
    ].to_numpy(dtype=float)

    high = plot_data.loc[
        (plot_data["Cell_type"] == cell) &
        (plot_data["RiskGroup"] == "High"),
        "ssGSEA_score"
    ].to_numpy(dtype=float)


    # Low
    bp1 = ax.boxplot(
        low,
        positions=[i - width/2],
        widths=0.25,
        showfliers=False,
        patch_artist=True
    )

    # High
    bp2 = ax.boxplot(
        high,
        positions=[i + width/2],
        widths=0.25,
        showfliers=False,
        patch_artist=True
    )


    # individual samples
    x_low = rng.normal(
        i - width/2,
        0.025,
        len(low)
    )

    x_high = rng.normal(
        i + width/2,
        0.025,
        len(high)
    )


    ax.scatter(
        x_low,
        low,
        s=5,
        alpha=0.12
    )

    ax.scatter(
        x_high,
        high,
        s=5,
        alpha=0.12
    )


    # FDR
    fdr = immune_comparison.loc[
        immune_comparison["Cell_type"] == cell,
        "FDR"
    ].iloc[0]


    if fdr < 0.001:
        sig = "***"
    elif fdr < 0.01:
        sig = "**"
    elif fdr < 0.05:
        sig = "*"
    else:
        sig = "ns"


    ymax = max(
        low.max(),
        high.max()
    )

    ymin = min(
        low.min(),
        high.min()
    )

    yrange = ymax - ymin

    ax.text(
        i,
        ymax + 0.08 * yrange,
        sig,
        ha="center",
        va="bottom",
        fontsize=10
    )


ax.set_xticks(
    positions
)

ax.set_xticklabels(
    [
        x.replace("_", " ")
        for x in plot_cells
    ],
    rotation=45,
    ha="right"
)


ax.set_ylabel(
    "ssGSEA enrichment score",
    fontsize=12
)

ax.set_xlabel("")

ax.set_title(
    "Immune cell infiltration between high- and low-risk groups",
    fontsize=13
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ssGSEA_High_vs_Low_ImmuneCells.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    os.path.join(
        fig_dir,
        "ssGSEA_High_vs_Low_ImmuneCells.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.34
# RiskScore vs immune-cell correlation bubble plot
# ============================================================

corr_plot = immune_correlations.copy()


# 避免-log10(0)
corr_plot["FDR_plot"] = corr_plot["FDR"].clip(
    lower=1e-12
)

corr_plot["Significance"] = (
    -np.log10(
        corr_plot["FDR_plot"]
    )
)


# 按rho排序
corr_plot = corr_plot.sort_values(
    "Spearman_rho"
).reset_index(drop=True)


y_pos = np.arange(
    len(corr_plot)
)


fig, ax = plt.subplots(
    figsize=(7.5, 6.5)
)


sizes = (
    40
    +
    corr_plot["Significance"] * 45
)


ax.scatter(
    corr_plot["Spearman_rho"],
    y_pos,
    s=sizes,
    alpha=0.75
)


ax.axvline(
    0,
    linestyle="--",
    linewidth=1
)


ax.set_yticks(
    y_pos
)

ax.set_yticklabels(
    corr_plot["Cell_type"]
    .str.replace("_", " ")
)


ax.set_xlabel(
    "Spearman correlation with RiskScore",
    fontsize=12
)

ax.set_ylabel("")

ax.set_title(
    "Correlation between RiskScore and immune-cell infiltration",
    fontsize=13
)


# 标FDR
for i, row in corr_plot.iterrows():

    if row["FDR"] < 0.001:
        sig = "***"
    elif row["FDR"] < 0.01:
        sig = "**"
    elif row["FDR"] < 0.05:
        sig = "*"
    else:
        sig = ""

    if row["Spearman_rho"] >= 0:
        x_text = row["Spearman_rho"] + 0.015
        ha = "left"
    else:
        x_text = row["Spearman_rho"] - 0.015
        ha = "right"

    ax.text(
        x_text,
        i,
        sig,
        va="center",
        ha=ha,
        fontsize=10
    )


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ssGSEA_RiskScore_ImmuneCell_Correlation.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    os.path.join(
        fig_dir,
        "ssGSEA_RiskScore_ImmuneCell_Correlation.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.35
# Immune-cell correlation heatmap
# ============================================================

immune_score_only = immune_merge[
    immune_cols
].copy()


corr_matrix = immune_score_only.corr(
    method="spearman"
)


fig, ax = plt.subplots(
    figsize=(9, 8)
)


im = ax.imshow(
    corr_matrix.values,
    vmin=-1,
    vmax=1,
    aspect="auto"
)


ax.set_xticks(
    np.arange(
        len(corr_matrix.columns)
    )
)

ax.set_yticks(
    np.arange(
        len(corr_matrix.index)
    )
)


ax.set_xticklabels(
    [
        x.replace("_", " ")
        for x in corr_matrix.columns
    ],
    rotation=45,
    ha="right",
    fontsize=9
)


ax.set_yticklabels(
    [
        x.replace("_", " ")
        for x in corr_matrix.index
    ],
    fontsize=9
)


# 写相关系数
for i in range(
    len(corr_matrix.index)
):

    for j in range(
        len(corr_matrix.columns)
    ):

        ax.text(
            j,
            i,
            f"{corr_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=7
        )


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

cbar.set_label(
    "Spearman correlation",
    fontsize=10
)


ax.set_title(
    "Correlation among immune-cell infiltration scores",
    fontsize=13
)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ssGSEA_ImmuneCell_Correlation_Heatmap.png"
    ),
    dpi=600,
    bbox_inches="tight"
)


fig.savefig(
    os.path.join(
        fig_dir,
        "ssGSEA_ImmuneCell_Correlation_Heatmap.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
#免疫检查点基因分析

In [ ]:
# ============================================================
# Step11.36 Immune checkpoint gene panel
# ============================================================

checkpoint_genes = [
    # PD-1 / PD-L1 axis
    "PDCD1",
    "CD274",
    "PDCD1LG2",

    # inhibitory checkpoints
    "CTLA4",
    "LAG3",
    "TIGIT",
    "HAVCR2",
    "BTLA",
    "CD160",
    "VSIR",

    # co-stimulatory / activation-related
    "ICOS",
    "CD27",
    "CD28",
    "TNFRSF9",
    "TNFRSF4",
    "CD40",
    "CD40LG",

    # additional immune regulatory molecules
    "IDO1",
    "CD276",
    "VTCN1"
]

print("Checkpoint genes:", len(checkpoint_genes))

In [ ]:
# ============================================================
# Step11.37 Check checkpoint gene availability
# ============================================================

present_checkpoint = [
    g for g in checkpoint_genes
    if g in rna_immune.index
]

missing_checkpoint = [
    g for g in checkpoint_genes
    if g not in rna_immune.index
]

print("Present:", len(present_checkpoint))
print(present_checkpoint)

print("\nMissing:", len(missing_checkpoint))
print(missing_checkpoint)

In [ ]:
# ============================================================
# Step11.38 Extract checkpoint expression
# ============================================================

checkpoint_expr = (
    rna_immune
    .loc[present_checkpoint]
    .T
    .copy()
)

checkpoint_expr.index.name = "Sample_ID"

print("Checkpoint expression shape:")
print(checkpoint_expr.shape)

display(checkpoint_expr.head())

In [ ]:
# ============================================================
# Step11.39 Merge with RiskScore
# ============================================================

checkpoint_merge = (
    checkpoint_expr
    .reset_index()
    .merge(
        risk_immune.reset_index(),
        on="Sample_ID",
        how="inner"
    )
)

print("Merged shape:", checkpoint_merge.shape)

display(checkpoint_merge.head())

In [ ]:
# ============================================================
# Step11.40 High vs Low checkpoint comparison
# ============================================================

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

checkpoint_comparison = []

for gene in present_checkpoint:

    high = checkpoint_merge.loc[
        checkpoint_merge["RiskGroup"] == "High",
        gene
    ].dropna()

    low = checkpoint_merge.loc[
        checkpoint_merge["RiskGroup"] == "Low",
        gene
    ].dropna()

    stat, p = mannwhitneyu(
        high,
        low,
        alternative="two-sided"
    )

    checkpoint_comparison.append({
        "Gene": gene,
        "High_n": len(high),
        "Low_n": len(low),
        "High_mean": high.mean(),
        "Low_mean": low.mean(),
        "High_median": high.median(),
        "Low_median": low.median(),
        "High_minus_Low": high.median() - low.median(),
        "P_value": p
    })


checkpoint_comparison = pd.DataFrame(
    checkpoint_comparison
)

checkpoint_comparison["FDR"] = multipletests(
    checkpoint_comparison["P_value"],
    method="fdr_bh"
)[1]

checkpoint_comparison = (
    checkpoint_comparison
    .sort_values(
        ["FDR", "P_value"]
    )
    .reset_index(drop=True)
)

display(checkpoint_comparison)

In [ ]:
# ============================================================
# Step11.41 RiskScore correlation with checkpoint genes
# ============================================================

from scipy.stats import spearmanr

checkpoint_correlations = []

for gene in present_checkpoint:

    tmp = checkpoint_merge[
        ["RiskScore", gene]
    ].dropna()

    rho, p = spearmanr(
        tmp["RiskScore"],
        tmp[gene]
    )

    checkpoint_correlations.append({
        "Gene": gene,
        "N": len(tmp),
        "Spearman_rho": rho,
        "P_value": p
    })


checkpoint_correlations = pd.DataFrame(
    checkpoint_correlations
)

checkpoint_correlations["FDR"] = multipletests(
    checkpoint_correlations["P_value"],
    method="fdr_bh"
)[1]

checkpoint_correlations = (
    checkpoint_correlations
    .sort_values(
        ["FDR", "P_value"]
    )
    .reset_index(drop=True)
)

display(checkpoint_correlations)

In [ ]:
# ============================================================
# Step11.42 Save checkpoint results
# ============================================================

checkpoint_expr.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ImmuneCheckpoint_Expression_365.csv"
    )
)

checkpoint_merge.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ImmuneCheckpoint_With_RiskGroup.csv"
    ),
    index=False
)

checkpoint_comparison.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ImmuneCheckpoint_High_vs_Low.csv"
    ),
    index=False
)

checkpoint_correlations.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ImmuneCheckpoint_RiskScore_Correlation.csv"
    ),
    index=False
)

print("Checkpoint analysis results saved.")

In [ ]:
# ============================================================
# Step11.43 Immune-regulatory genes
# High vs Low summary plot
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# 按 FDR 排序
plot_genes = (
    checkpoint_comparison
    .sort_values("FDR")
    ["Gene"]
    .tolist()
)


# 提取中位数差值
plot_df = (
    checkpoint_comparison
    .set_index("Gene")
    .loc[plot_genes]
    .reset_index()
)


plot_df["Direction"] = np.where(
    plot_df["High_minus_Low"] > 0,
    "Higher in High risk",
    "Lower in High risk"
)


# -log10(FDR)
plot_df["minus_log10_FDR"] = (
    -np.log10(
        plot_df["FDR"].clip(lower=1e-12)
    )
)


y = np.arange(
    len(plot_df)
)


fig, ax = plt.subplots(
    figsize=(8, 8)
)


ax.scatter(
    plot_df["High_minus_Low"],
    y,
    s=40 + plot_df["minus_log10_FDR"] * 35,
    alpha=0.75
)


ax.axvline(
    0,
    linestyle="--",
    linewidth=1
)


ax.set_yticks(y)

ax.set_yticklabels(
    plot_df["Gene"]
)


ax.set_xlabel(
    "Median expression difference (High − Low)",
    fontsize=12
)


ax.set_ylabel("")

ax.set_title(
    "Immune-regulatory gene expression differences",
    fontsize=13
)


for i, row in plot_df.iterrows():

    if row["FDR"] < 0.001:
        sig = "***"
    elif row["FDR"] < 0.01:
        sig = "**"
    elif row["FDR"] < 0.05:
        sig = "*"
    else:
        sig = ""

    if row["High_minus_Low"] >= 0:
        xpos = row["High_minus_Low"] + 0.02
        ha = "left"
    else:
        xpos = row["High_minus_Low"] - 0.02
        ha = "right"

    ax.text(
        xpos,
        i,
        sig,
        va="center",
        ha=ha,
        fontsize=10
    )


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneCheckpoint_High_vs_Low_Summary.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneCheckpoint_High_vs_Low_Summary.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.44
# RiskScore vs immune-regulatory genes
# ============================================================

corr_df = (
    checkpoint_correlations
    .sort_values("Spearman_rho")
    .reset_index(drop=True)
    .copy()
)


corr_df["minus_log10_FDR"] = (
    -np.log10(
        corr_df["FDR"].clip(lower=1e-12)
    )
)


y = np.arange(
    len(corr_df)
)


fig, ax = plt.subplots(
    figsize=(8, 8)
)


ax.scatter(
    corr_df["Spearman_rho"],
    y,
    s=40 + corr_df["minus_log10_FDR"] * 35,
    alpha=0.75
)


ax.axvline(
    0,
    linestyle="--",
    linewidth=1
)


ax.set_yticks(y)

ax.set_yticklabels(
    corr_df["Gene"]
)


ax.set_xlabel(
    "Spearman correlation with RiskScore",
    fontsize=12
)

ax.set_ylabel("")

ax.set_title(
    "Correlation between RiskScore and immune-regulatory genes",
    fontsize=13
)


for i, row in corr_df.iterrows():

    if row["FDR"] < 0.001:
        sig = "***"
    elif row["FDR"] < 0.01:
        sig = "**"
    elif row["FDR"] < 0.05:
        sig = "*"
    else:
        sig = ""

    if row["Spearman_rho"] >= 0:
        xpos = row["Spearman_rho"] + 0.015
        ha = "left"
    else:
        xpos = row["Spearman_rho"] - 0.015
        ha = "right"

    ax.text(
        xpos,
        i,
        sig,
        va="center",
        ha=ha,
        fontsize=10
    )


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneCheckpoint_RiskScore_Correlation.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneCheckpoint_RiskScore_Correlation.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.45 Focus checkpoint boxplots
# ============================================================

focus_genes = [
    "TNFRSF4",
    "CD276",
    "VTCN1",
    "CD40LG",
    "BTLA",
    "CD160"
]


fig, axes = plt.subplots(
    2,
    3,
    figsize=(13, 8)
)

axes = axes.flatten()

rng = np.random.default_rng(123)


for ax, gene in zip(
    axes,
    focus_genes
):

    low = checkpoint_merge.loc[
        checkpoint_merge["RiskGroup"] == "Low",
        gene
    ].to_numpy(dtype=float)

    high = checkpoint_merge.loc[
        checkpoint_merge["RiskGroup"] == "High",
        gene
    ].to_numpy(dtype=float)


    fdr = checkpoint_comparison.loc[
        checkpoint_comparison["Gene"] == gene,
        "FDR"
    ].iloc[0]


    ax.boxplot(
        [low, high],
        tick_labels=[
            "Low risk",
            "High risk"
        ],
        widths=0.55,
        showfliers=False
    )


    for i, values in enumerate(
        [low, high],
        start=1
    ):

        x = rng.normal(
            i,
            0.04,
            len(values)
        )

        ax.scatter(
            x,
            values,
            s=7,
            alpha=0.20
        )


    if fdr < 0.001:
        sig = "***"
    elif fdr < 0.01:
        sig = "**"
    elif fdr < 0.05:
        sig = "*"
    else:
        sig = "ns"


    ax.set_title(
        f"{gene} ({sig})",
        fontsize=12
    )

    ax.set_ylabel(
        "log2(CPM + 1)"
    )


    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneCheckpoint_FocusGenes_Boxplots.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneCheckpoint_FocusGenes_Boxplots.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.46 Focus genes vs RiskScore
# ============================================================

from scipy.stats import spearmanr


fig, axes = plt.subplots(
    2,
    3,
    figsize=(13, 8)
)

axes = axes.flatten()


for ax, gene in zip(
    axes,
    focus_genes
):

    x = checkpoint_merge[
        "RiskScore"
    ].to_numpy(dtype=float)

    y = checkpoint_merge[
        gene
    ].to_numpy(dtype=float)


    rho, p = spearmanr(
        x,
        y
    )


    ax.scatter(
        x,
        y,
        s=15,
        alpha=0.45
    )


    coef = np.polyfit(
        x,
        y,
        1
    )

    xline = np.linspace(
        x.min(),
        x.max(),
        200
    )

    yline = (
        coef[0] * xline
        +
        coef[1]
    )


    ax.plot(
        xline,
        yline,
        linewidth=2
    )


    if p < 0.001:
        ptxt = "P < 0.001"
    else:
        ptxt = f"P = {p:.3f}"


    ax.text(
        0.05,
        0.95,
        f"ρ = {rho:.3f}\n{ptxt}",
        transform=ax.transAxes,
        va="top",
        fontsize=10
    )


    ax.set_title(
        gene,
        fontsize=12
    )

    ax.set_xlabel(
        "RiskScore"
    )

    ax.set_ylabel(
        "Expression"
    )


    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneCheckpoint_FocusGenes_RiskScore.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneCheckpoint_FocusGenes_RiskScore.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.47 Immune functional signatures
# ============================================================

immune_function_signatures = {

    "Cytolytic_activity": [
        "GZMA", "GZMB", "GZMH", "GZMK",
        "PRF1", "NKG7", "GNLY", "CTSW"
    ],

    "T_cell_activation": [
        "CD3D", "CD3E", "CD3G",
        "CD247", "CD69", "IL2RA",
        "ICOS", "CD28"
    ],

    "T_cell_exhaustion": [
        "PDCD1", "CTLA4", "LAG3",
        "TIGIT", "HAVCR2", "TOX",
        "CXCL13", "ENTPD1"
    ],

    "Antigen_presentation": [
        "B2M", "TAP1", "TAP2",
        "TAPBP", "PSMB8", "PSMB9",
        "NLRC5"
    ],

    "HLA_activity": [
        "HLA-A", "HLA-B", "HLA-C",
        "HLA-DPA1", "HLA-DPB1",
        "HLA-DQA1", "HLA-DQB1",
        "HLA-DRA", "HLA-DRB1"
    ],

    "Type_I_IFN_response": [
        "IFIT1", "IFIT2", "IFIT3",
        "ISG15", "MX1", "MX2",
        "OAS1", "OAS2", "OAS3"
    ],

    "Type_II_IFN_response": [
        "IFNG", "STAT1", "IRF1",
        "CXCL9", "CXCL10",
        "GBP1", "GBP2"
    ],

    "APC_co_stimulation": [
        "CD40", "CD80", "CD86",
        "ICOSLG", "TNFSF4",
        "TNFSF9"
    ],

    "APC_co_inhibition": [
        "CD274", "PDCD1LG2",
        "LGALS9", "VSIR",
        "VTCN1", "CD276"
    ],

    "Inflammation_promoting": [
        "IL1B", "TNF", "CXCL8",
        "CCL2", "CCL3", "CCL4",
        "NFKB1", "RELA"
    ],

    "Checkpoint_activity": [
        "PDCD1", "CD274", "CTLA4",
        "LAG3", "TIGIT", "HAVCR2",
        "BTLA", "CD276", "VTCN1"
    ],

    "Parainflammation": [
        "STAT1", "STAT3", "IRF1",
        "IRF7", "NFKB1", "RELA",
        "CXCL10", "IL6"
    ]
}


print("Number of immune functional signatures:",
      len(immune_function_signatures))

In [ ]:
# ============================================================
# Step11.48 Signature coverage QC
# ============================================================

function_coverage = []

for pathway, genes in immune_function_signatures.items():

    present = [g for g in genes if g in rna_immune.index]
    missing = [g for g in genes if g not in rna_immune.index]

    function_coverage.append({
        "Function": pathway,
        "Signature_genes": len(genes),
        "Present_genes": len(present),
        "Missing_genes": len(missing),
        "Coverage": len(present) / len(genes)
    })


function_coverage_df = pd.DataFrame(
    function_coverage
)

display(function_coverage_df)

In [ ]:
function_coverage_df.to_csv(
    os.path.join(
        result_dir,
        "ImmuneFunction_Signature_Coverage.csv"
    ),
    index=False
)

In [ ]:
# ============================================================
# Step11.49 Run immune-function ssGSEA
# ============================================================

function_ssgsea = gp.ssgsea(
    data=rna_immune,
    gene_sets=immune_function_signatures,
    sample_norm_method="rank",
    outdir=None,
    no_plot=True,
    min_size=2,
    max_size=500,
    permutation_num=0,
    threads=4,
    seed=123,
    verbose=True
)

In [ ]:
print(function_ssgsea.res2d.shape)
print(function_ssgsea.res2d.columns.tolist())

display(function_ssgsea.res2d.head())

In [ ]:
# ============================================================
# Step11.50 Build sample × immune-function matrix
# ============================================================

function_long = function_ssgsea.res2d.copy()

function_long["NES"] = pd.to_numeric(
    function_long["NES"],
    errors="coerce"
)

function_matrix = (
    function_long
    .pivot(
        index="Name",
        columns="Term",
        values="NES"
    )
)

function_matrix.index.name = "Sample_ID"


print("Immune function matrix:")
print(function_matrix.shape)

print("\nFunctions:")
print(function_matrix.columns.tolist())

display(function_matrix.head())

In [ ]:
# ============================================================
# Step11.51 Save immune-function scores
# ============================================================

function_matrix.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ssGSEA_Immune_Function_Scores_365.csv"
    )
)

function_long.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ssGSEA_Immune_Function_Long.csv"
    ),
    index=False
)

print("Immune-function ssGSEA results saved.")

In [ ]:
# ============================================================
# Step11.52 Merge with RiskGroup
# ============================================================

function_matrix.index.name = "Sample_ID"
risk_immune.index.name = "Sample_ID"

function_merge = (
    function_matrix
    .reset_index()
    .merge(
        risk_immune.reset_index(),
        on="Sample_ID",
        how="inner"
    )
)

print(function_merge.shape)

In [ ]:
# ============================================================
# Step11.53 High vs Low immune-function comparison
# ============================================================

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

function_cols = function_matrix.columns.tolist()

function_comparison = []

for func in function_cols:

    high = function_merge.loc[
        function_merge["RiskGroup"] == "High",
        func
    ].dropna()

    low = function_merge.loc[
        function_merge["RiskGroup"] == "Low",
        func
    ].dropna()

    stat, p = mannwhitneyu(
        high,
        low,
        alternative="two-sided"
    )

    function_comparison.append({
        "Function": func,
        "High_n": len(high),
        "Low_n": len(low),
        "High_mean": high.mean(),
        "Low_mean": low.mean(),
        "High_median": high.median(),
        "Low_median": low.median(),
        "High_minus_Low": high.median() - low.median(),
        "P_value": p
    })


function_comparison = pd.DataFrame(
    function_comparison
)

function_comparison["FDR"] = multipletests(
    function_comparison["P_value"],
    method="fdr_bh"
)[1]

function_comparison = (
    function_comparison
    .sort_values(["FDR", "P_value"])
    .reset_index(drop=True)
)

display(function_comparison)

In [ ]:
# ============================================================
# Step11.54 RiskScore correlation with immune functions
# ============================================================

from scipy.stats import spearmanr

function_correlations = []

for func in function_cols:

    tmp = function_merge[
        ["RiskScore", func]
    ].dropna()

    rho, p = spearmanr(
        tmp["RiskScore"],
        tmp[func]
    )

    function_correlations.append({
        "Function": func,
        "N": len(tmp),
        "Spearman_rho": rho,
        "P_value": p
    })


function_correlations = pd.DataFrame(
    function_correlations
)

function_correlations["FDR"] = multipletests(
    function_correlations["P_value"],
    method="fdr_bh"
)[1]

function_correlations = (
    function_correlations
    .sort_values(["FDR", "P_value"])
    .reset_index(drop=True)
)

display(function_correlations)

In [ ]:
function_comparison.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ImmuneFunction_High_vs_Low.csv"
    ),
    index=False
)

function_correlations.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ImmuneFunction_RiskScore_Correlation.csv"
    ),
    index=False
)

function_merge.to_csv(
    os.path.join(
        result_dir,
        "LIHC_ImmuneFunction_With_RiskGroup.csv"
    ),
    index=False
)

print("Immune-function statistics saved.")

In [ ]:
# ============================================================
# Step11.55
# High vs Low immune-function comparison
# ============================================================

import numpy as np
import matplotlib.pyplot as plt


plot_functions = (
    function_comparison
    .sort_values("FDR")
    ["Function"]
    .tolist()
)


fig, ax = plt.subplots(
    figsize=(15, 7)
)

positions = np.arange(
    len(plot_functions)
)

width = 0.30

rng = np.random.default_rng(123)


for i, func in enumerate(plot_functions):

    low = function_merge.loc[
        function_merge["RiskGroup"] == "Low",
        func
    ].to_numpy(dtype=float)

    high = function_merge.loc[
        function_merge["RiskGroup"] == "High",
        func
    ].to_numpy(dtype=float)


    ax.boxplot(
        low,
        positions=[i - width/2],
        widths=0.25,
        showfliers=False
    )

    ax.boxplot(
        high,
        positions=[i + width/2],
        widths=0.25,
        showfliers=False
    )


    # jitter points
    x_low = rng.normal(
        i - width/2,
        0.025,
        len(low)
    )

    x_high = rng.normal(
        i + width/2,
        0.025,
        len(high)
    )


    ax.scatter(
        x_low,
        low,
        s=5,
        alpha=0.12
    )

    ax.scatter(
        x_high,
        high,
        s=5,
        alpha=0.12
    )


    # significance
    fdr = function_comparison.loc[
        function_comparison["Function"] == func,
        "FDR"
    ].iloc[0]


    if fdr < 0.001:
        sig = "***"
    elif fdr < 0.01:
        sig = "**"
    elif fdr < 0.05:
        sig = "*"
    else:
        sig = "ns"


    ymax = max(
        low.max(),
        high.max()
    )

    ymin = min(
        low.min(),
        high.min()
    )

    yrange = ymax - ymin


    ax.text(
        i,
        ymax + 0.08 * yrange,
        sig,
        ha="center",
        fontsize=10
    )


ax.set_xticks(
    positions
)

ax.set_xticklabels(
    [
        x.replace("_", " ")
        for x in plot_functions
    ],
    rotation=45,
    ha="right"
)


ax.set_ylabel(
    "ssGSEA enrichment score",
    fontsize=12
)

ax.set_title(
    "Immune functional activity between high- and low-risk groups",
    fontsize=13
)


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_High_vs_Low_Combined.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_High_vs_Low_Combined.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.56
# RiskScore vs immune-function correlation
# ============================================================

corr_func = (
    function_correlations
    .sort_values("Spearman_rho")
    .reset_index(drop=True)
    .copy()
)


corr_func["minus_log10_FDR"] = (
    -np.log10(
        corr_func["FDR"].clip(lower=1e-12)
    )
)


y = np.arange(
    len(corr_func)
)


fig, ax = plt.subplots(
    figsize=(8, 7)
)


sizes = (
    40
    +
    corr_func["minus_log10_FDR"] * 45
)


ax.scatter(
    corr_func["Spearman_rho"],
    y,
    s=sizes,
    alpha=0.75
)


ax.axvline(
    0,
    linestyle="--",
    linewidth=1
)


ax.set_yticks(y)

ax.set_yticklabels(
    corr_func["Function"]
    .str.replace("_", " ")
)


ax.set_xlabel(
    "Spearman correlation with RiskScore",
    fontsize=12
)

ax.set_ylabel("")

ax.set_title(
    "Correlation between RiskScore and immune functions",
    fontsize=13
)


for i, row in corr_func.iterrows():

    if row["FDR"] < 0.001:
        sig = "***"
    elif row["FDR"] < 0.01:
        sig = "**"
    elif row["FDR"] < 0.05:
        sig = "*"
    else:
        sig = ""

    if row["Spearman_rho"] >= 0:
        xpos = row["Spearman_rho"] + 0.012
        ha = "left"
    else:
        xpos = row["Spearman_rho"] - 0.012
        ha = "right"

    ax.text(
        xpos,
        i,
        sig,
        va="center",
        ha=ha,
        fontsize=10
    )


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_RiskScore_Correlation.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_RiskScore_Correlation.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.57
# Immune-function correlation heatmap
# ============================================================

function_score_only = function_merge[
    function_cols
].copy()


function_corr_matrix = (
    function_score_only
    .corr(
        method="spearman"
    )
)


fig, ax = plt.subplots(
    figsize=(10, 9)
)


im = ax.imshow(
    function_corr_matrix.values,
    vmin=-1,
    vmax=1,
    aspect="auto"
)


ax.set_xticks(
    np.arange(
        len(function_corr_matrix.columns)
    )
)

ax.set_yticks(
    np.arange(
        len(function_corr_matrix.index)
    )
)


ax.set_xticklabels(
    [
        x.replace("_", " ")
        for x in function_corr_matrix.columns
    ],
    rotation=45,
    ha="right",
    fontsize=9
)


ax.set_yticklabels(
    [
        x.replace("_", " ")
        for x in function_corr_matrix.index
    ],
    fontsize=9
)


for i in range(
    len(function_corr_matrix.index)
):

    for j in range(
        len(function_corr_matrix.columns)
    ):

        ax.text(
            j,
            i,
            f"{function_corr_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=7
        )


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

cbar.set_label(
    "Spearman correlation",
    fontsize=10
)


ax.set_title(
    "Correlation among immune functional signatures",
    fontsize=13
)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_Correlation_Heatmap.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_Correlation_Heatmap.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.58
# Focus immune-function boxplots
# ============================================================

focus_functions = [
    "Type_I_IFN_response",
    "Cytolytic_activity",
    "Type_II_IFN_response",
    "Parainflammation",
    "T_cell_activation"
]


fig, axes = plt.subplots(
    2,
    3,
    figsize=(14, 8)
)

axes = axes.flatten()

rng = np.random.default_rng(123)


for ax, func in zip(
    axes,
    focus_functions
):

    low = function_merge.loc[
        function_merge["RiskGroup"] == "Low",
        func
    ].to_numpy(dtype=float)

    high = function_merge.loc[
        function_merge["RiskGroup"] == "High",
        func
    ].to_numpy(dtype=float)


    fdr = function_comparison.loc[
        function_comparison["Function"] == func,
        "FDR"
    ].iloc[0]


    ax.boxplot(
        [low, high],
        tick_labels=[
            "Low risk",
            "High risk"
        ],
        widths=0.55,
        showfliers=False
    )


    for i, values in enumerate(
        [low, high],
        start=1
    ):

        x = rng.normal(
            i,
            0.04,
            len(values)
        )

        ax.scatter(
            x,
            values,
            s=7,
            alpha=0.20
        )


    if fdr < 0.001:
        sig = "***"
    elif fdr < 0.01:
        sig = "**"
    elif fdr < 0.05:
        sig = "*"
    else:
        sig = "ns"


    ax.set_title(
        f"{func.replace('_',' ')} ({sig})",
        fontsize=11
    )

    ax.set_ylabel(
        "ssGSEA score"
    )


    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# 删除空白子图
for j in range(
    len(focus_functions),
    len(axes)
):
    fig.delaxes(
        axes[j]
    )


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_Focus_Boxplots.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_Focus_Boxplots.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.59
# Focus immune functions vs RiskScore
# ============================================================

from scipy.stats import spearmanr


fig, axes = plt.subplots(
    2,
    3,
    figsize=(14, 8)
)

axes = axes.flatten()


for ax, func in zip(
    axes,
    focus_functions
):

    x = function_merge[
        "RiskScore"
    ].to_numpy(dtype=float)

    y = function_merge[
        func
    ].to_numpy(dtype=float)


    rho, p = spearmanr(
        x,
        y
    )


    ax.scatter(
        x,
        y,
        s=15,
        alpha=0.45
    )


    coef = np.polyfit(
        x,
        y,
        1
    )

    xline = np.linspace(
        x.min(),
        x.max(),
        200
    )

    yline = (
        coef[0] * xline
        +
        coef[1]
    )


    ax.plot(
        xline,
        yline,
        linewidth=2
    )


    if p < 0.001:
        ptxt = "P < 0.001"
    else:
        ptxt = f"P = {p:.3f}"


    ax.text(
        0.05,
        0.95,
        f"ρ = {rho:.3f}\n{ptxt}",
        transform=ax.transAxes,
        va="top",
        fontsize=10
    )


    ax.set_title(
        func.replace("_", " "),
        fontsize=11
    )

    ax.set_xlabel(
        "RiskScore"
    )

    ax.set_ylabel(
        "ssGSEA score"
    )


    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


for j in range(
    len(focus_functions),
    len(axes)
):
    fig.delaxes(
        axes[j]
    )


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_Focus_RiskScore.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_Focus_RiskScore.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.60 Focus immune functions
# Boxplots + RiskScore scatterplots
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

focus_functions = [
    "Type_I_IFN_response",
    "Cytolytic_activity",
    "Type_II_IFN_response",
    "Parainflammation",
    "T_cell_activation"
]

# ---------- A. Boxplots ----------
fig, axes = plt.subplots(
    2,
    3,
    figsize=(14, 8)
)

axes = axes.flatten()

rng = np.random.default_rng(123)

for ax, func in zip(axes, focus_functions):

    low = function_merge.loc[
        function_merge["RiskGroup"] == "Low",
        func
    ].to_numpy(dtype=float)

    high = function_merge.loc[
        function_merge["RiskGroup"] == "High",
        func
    ].to_numpy(dtype=float)

    fdr = function_comparison.loc[
        function_comparison["Function"] == func,
        "FDR"
    ].iloc[0]

    ax.boxplot(
        [low, high],
        tick_labels=["Low risk", "High risk"],
        widths=0.55,
        showfliers=False
    )

    for i, values in enumerate([low, high], start=1):
        x = rng.normal(i, 0.04, len(values))
        ax.scatter(
            x,
            values,
            s=8,
            alpha=0.20
        )

    if fdr < 0.001:
        sig = "***"
    elif fdr < 0.01:
        sig = "**"
    elif fdr < 0.05:
        sig = "*"
    else:
        sig = "ns"

    ax.set_title(
        f"{func.replace('_', ' ')} ({sig})",
        fontsize=11
    )

    ax.set_ylabel("ssGSEA score")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for j in range(len(focus_functions), len(axes)):
    fig.delaxes(axes[j])

fig.tight_layout()

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_Focus_Boxplots.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_Focus_Boxplots.pdf"
    ),
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# ---------- B. Scatterplots ----------
fig, axes = plt.subplots(
    2,
    3,
    figsize=(14, 8)
)

axes = axes.flatten()

for ax, func in zip(axes, focus_functions):

    x = function_merge["RiskScore"].to_numpy(dtype=float)
    y = function_merge[func].to_numpy(dtype=float)

    rho, p = spearmanr(x, y)

    ax.scatter(
        x,
        y,
        s=16,
        alpha=0.45
    )

    coef = np.polyfit(x, y, 1)

    xline = np.linspace(
        x.min(),
        x.max(),
        200
    )

    yline = (
        coef[0] * xline
        +
        coef[1]
    )

    ax.plot(
        xline,
        yline,
        linewidth=2
    )

    if p < 0.001:
        ptxt = "P < 0.001"
    else:
        ptxt = f"P = {p:.3f}"

    ax.text(
        0.05,
        0.95,
        f"ρ = {rho:.3f}\n{ptxt}",
        transform=ax.transAxes,
        va="top",
        fontsize=10
    )

    ax.set_title(
        func.replace("_", " "),
        fontsize=11
    )

    ax.set_xlabel("RiskScore")
    ax.set_ylabel("ssGSEA score")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for j in range(len(focus_functions), len(axes)):
    fig.delaxes(axes[j])

fig.tight_layout()

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_Focus_RiskScore.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "ImmuneFunction_Focus_RiskScore.pdf"
    ),
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.61 Integrated immune correlation heatmap
# Immune cells + immune functions + key checkpoint genes
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# 1. 免疫细胞
immune_part = ssgsea_matrix.copy()

immune_part.columns = [
    "Cell_" + x
    for x in immune_part.columns
]


# 2. 免疫功能
function_part = function_matrix.copy()

function_part.columns = [
    "Function_" + x
    for x in function_part.columns
]


# 3. 重点checkpoint
key_checkpoint_genes = [
    "TNFRSF4",
    "CD276",
    "VTCN1",
    "CD40LG",
    "BTLA",
    "CD160"
]

checkpoint_part = checkpoint_expr[
    key_checkpoint_genes
].copy()

checkpoint_part.columns = [
    "Checkpoint_" + x
    for x in checkpoint_part.columns
]


# 4. 对齐样本
common_samples = (
    immune_part.index
    .intersection(function_part.index)
    .intersection(checkpoint_part.index)
)

immune_integrated = pd.concat(
    [
        immune_part.loc[common_samples],
        function_part.loc[common_samples],
        checkpoint_part.loc[common_samples]
    ],
    axis=1
)


print(
    "Integrated matrix:",
    immune_integrated.shape
)

display(
    immune_integrated.iloc[:5, :8]
)

In [ ]:
# ============================================================
# Step11.62 Spearman correlation matrix
# ============================================================

integrated_corr = immune_integrated.corr(
    method="spearman"
)

print(
    integrated_corr.shape
)

In [ ]:
# ============================================================
# Step11.63 Full integrated correlation heatmap
# ============================================================

fig, ax = plt.subplots(
    figsize=(16, 14)
)

im = ax.imshow(
    integrated_corr.values,
    vmin=-1,
    vmax=1,
    aspect="auto"
)

labels = (
    integrated_corr.columns
    .str.replace("Cell_", "", regex=False)
    .str.replace("Function_", "", regex=False)
    .str.replace("Checkpoint_", "", regex=False)
    .str.replace("_", " ", regex=False)
)

ax.set_xticks(
    np.arange(
        len(labels)
    )
)

ax.set_yticks(
    np.arange(
        len(labels)
    )
)

ax.set_xticklabels(
    labels,
    rotation=90,
    fontsize=7
)

ax.set_yticklabels(
    labels,
    fontsize=7
)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.035,
    pad=0.02
)

cbar.set_label(
    "Spearman correlation",
    fontsize=10
)

ax.set_title(
    "Integrated immune microenvironment correlation landscape",
    fontsize=14
)

fig.tight_layout()

fig.savefig(
    os.path.join(
        fig_dir,
        "Integrated_Immune_Correlation_Heatmap.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "Integrated_Immune_Correlation_Heatmap.pdf"
    ),
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.64 Cross-layer immune correlation heatmap
# Rows: immune cells
# Columns: immune functions + checkpoints
# ============================================================

row_cols = [
    c for c in immune_integrated.columns
    if c.startswith("Cell_")
]

col_cols = [
    c for c in immune_integrated.columns
    if c.startswith("Function_")
    or c.startswith("Checkpoint_")
]


cross_corr = pd.DataFrame(
    index=row_cols,
    columns=col_cols,
    dtype=float
)


for row in row_cols:
    for col in col_cols:

        cross_corr.loc[row, col] = (
            immune_integrated[
                [row, col]
            ]
            .corr(
                method="spearman"
            )
            .iloc[0, 1]
        )


fig, ax = plt.subplots(
    figsize=(14, 8)
)


im = ax.imshow(
    cross_corr.values,
    vmin=-1,
    vmax=1,
    aspect="auto"
)


xlabels = [
    x.replace("Function_", "")
     .replace("Checkpoint_", "")
     .replace("_", " ")
    for x in cross_corr.columns
]

ylabels = [
    x.replace("Cell_", "")
     .replace("_", " ")
    for x in cross_corr.index
]


ax.set_xticks(
    np.arange(
        len(xlabels)
    )
)

ax.set_yticks(
    np.arange(
        len(ylabels)
    )
)

ax.set_xticklabels(
    xlabels,
    rotation=60,
    ha="right",
    fontsize=8
)

ax.set_yticklabels(
    ylabels,
    fontsize=9
)


# 写数字
for i in range(
    cross_corr.shape[0]
):
    for j in range(
        cross_corr.shape[1]
    ):

        value = cross_corr.iloc[i, j]

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=6
        )


cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.035,
    pad=0.02
)

cbar.set_label(
    "Spearman correlation",
    fontsize=10
)


ax.set_title(
    "Cross-layer correlation of immune cells, functions and regulatory genes",
    fontsize=13
)


fig.tight_layout()


fig.savefig(
    os.path.join(
        fig_dir,
        "Integrated_Immune_CrossLayer_Heatmap.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    os.path.join(
        fig_dir,
        "Integrated_Immune_CrossLayer_Heatmap.pdf"
    ),
    bbox_inches="tight"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Step11.65 Save integrated immune matrices
# ============================================================

immune_integrated.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Integrated_Immune_Matrix_365.csv"
    )
)

integrated_corr.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Integrated_Immune_Correlation_Matrix.csv"
    )
)

cross_corr.to_csv(
    os.path.join(
        result_dir,
        "LIHC_Integrated_Immune_CrossLayer_Correlation.csv"
    )
)

print("Integrated immune analysis saved.")